# 日频异常收益（AR）大表

`build/car.parquet` 只在公告事件的三个锚点上取值，中间的逐日收益用完就扔了。
这里把它补成一张 **permno × 交易日** 的全市场日频面板，之后换任何事件日
（公告日、新闻日、任意日期）、任何窗口都不必再扫一遍收益。

**输出** `build/ar_daily/ar_daily_YYYY.parquet`，一年一个文件

| 列 | 含义 |
|---|---|
| `permno` `date` | 公司、日历日 |
| `td_idx` | 交易日历下标，0–7672（`cal[td_idx]` 即 `date`）。与 `pead_panel.td0_idx` 同一套 |
| `port25` | **该行日期当天**所属的 size×B/M 组合，1–25，**0 表示当年没归上组** |
| `ret_c2c` `ret_o2o` | 个股日收益 |
| `bench_c2c` `bench_o2o` | 同组合当日等权收益 |
| `ar_c2c` `ar_o2o` | $AR_{i,t}=R_{i,t}-R_{p(i),t}$ |

`bench_*` 存了冗余（只有每天 25 个取值），但 `ret` 缺失的行推不出 `ret − ar`，
而 O2O 在 2008–2018 有两成行没有收益，所以直接存下来。

**上游全部现成，一行都不重算**

| 来源 | 用途 |
|---|---|
| `data/crsp_daily_YYYY.parquet` 的 `dlyret` | C2C 日收益 |
| `export/crsp_daily_ret_c2c_o2o.parquet` 的 `O2O_RET`（yifei v5） | O2O 日收益 |
| `build/port25_membership.parquet` | `(permno, ffyear) → port25`，成员一年一排，7 月至次年 6 月不变 |
| `build/port25_bench_returns.parquet` | `(date, port25) → 基准日收益`，C2C / O2O 各一套 |

C2C 必须取 `data/crsp_daily_*.parquet`：C2C 基准组合就是从这份建的。export 面板里的
`RET` 来自老 dsf pickle，数值一致（差 ≤1e-6）但行集少几百行，混用会让个别事件的窗口收益差到几个百分点。

**一处口径提醒**：这里的 `ar` 是**逐日算术差**。`car.parquet` 里的 CAR 是
buy-and-hold（连乘后相减），逐日 AR 加总 ≠ 那个 CAR。短窗几乎一样，60 天窗口能差几十个 bp。
要复现 `car_ann/car_drift` 的口径，用本表的 `ret_*` 连乘再减基准连乘，见最后一格的校验。

**组合归属**：逐日 AR 用**当日**所在 formation 年的 `port25`，每年 7 月 1 日换组；
`car.parquet` 那边是**事件时点固定**、整个窗口不换组。算长窗口 CAR 时按后者做。

**用 `td0_idx` 的坑**：`pead_panel` 里 1995 年公告的 5,628 个事件（`flag_pre_calendar`）
`td0` 全被塌到日历首日 1996-01-02，`td0_idx` 全是 0。原来的 CAR 因为整行置空才没出事，
新算的东西不会自动继承这个保护 —— 拿 `td0_idx` 取窗口前先 `~flag_pre_calendar`。


In [7]:
import datetime
import glob
import os
import shutil

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

DATA, BUILD, EXPORT = "data", "build", "export"
AR_DIR = f"{BUILD}/ar_daily"
RET_PANEL = f"{EXPORT}/crsp_daily_ret_c2c_o2o.parquet"

FORCE = True        # True 则忽略已有产物重建

# 交易日历：与 pead_panel 的 td0_idx 同一套
_d = [pd.read_parquet(p, columns=["dlycaldt"])["dlycaldt"].drop_duplicates()
      for p in sorted(glob.glob(f"{DATA}/crsp_daily_*.parquet"))]
cal = pd.to_datetime(pd.concat(_d)).drop_duplicates().sort_values().reset_index(drop=True)
del _d
cal_map = pd.Series(np.arange(len(cal), dtype="int32"), index=cal.values)
NDAY = len(cal)
print(f"交易日历 {NDAY:,} 天: {cal.iloc[0].date()} ~ {cal.iloc[-1].date()}")


交易日历 7,673 天: 1996-01-02 ~ 2026-06-30


In [8]:
def ffyear_of(dates):
    """formation 年：7 月至次年 6 月归属同一个，与 Step 2 建组合时一致。"""
    y = dates.dt.year
    return np.where(dates.dt.month >= 7, y, y - 1)


# ① (permno, ffyear) → port25
mem = pd.read_parquet(f"{BUILD}/port25_membership.parquet",
                      columns=["permno", "ffyear", "port25"])
mem = mem.astype({"permno": "int64", "ffyear": "int32", "port25": "int8"})

# ② (port25, td_idx) → 基准组合当日等权收益，C2C / O2O 各一套。
#    做成稠密矩阵，按下标取值，不用逐年 merge
bench = pd.read_parquet(f"{BUILD}/port25_bench_returns.parquet")
bench["td_idx"] = pd.to_datetime(bench["date"]).map(cal_map)
bench = bench.dropna(subset=["td_idx"])
bench["td_idx"] = bench["td_idx"].astype(int)
BRET = {}
for tag, col in [("c2c", "bench_ret_c2c"), ("o2o", "bench_ret_o2o")]:
    m = np.full((26, NDAY), np.nan)                          # 0 行留空，port25 取 1..25
    b = bench.dropna(subset=[col])
    m[b["port25"].to_numpy().astype(int), b["td_idx"].to_numpy()] = b[col].to_numpy()
    BRET[tag] = m
    print(f"基准 {tag}: {int((~np.isnan(m)).sum()):,} 个 (组合, 日)")

print(f"port25 成员 {len(mem):,} 条, formation 年 {mem['ffyear'].min()}–{mem['ffyear'].max()}")


基准 c2c: 191,825 个 (组合, 日)
基准 o2o: 190,225 个 (组合, 日)
port25 成员 135,230 条, formation 年 1995–2026


In [9]:
SCHEMA = pa.schema([
    ("permno", pa.int32()),
    ("date", pa.timestamp("s")),
    ("td_idx", pa.int32()),
    ("port25", pa.int8()),
    ("ret_c2c", pa.float32()),
    ("ret_o2o", pa.float32()),
    ("bench_c2c", pa.float32()),
    ("bench_o2o", pa.float32()),
    ("ar_c2c", pa.float32()),
    ("ar_o2o", pa.float32()),
])
YEARS = list(range(cal.iloc[0].year, cal.iloc[-1].year + 1))

if glob.glob(f"{AR_DIR}/ar_daily_*.parquet") and not FORCE:
    print(f"[缓存] {AR_DIR}")
else:
    shutil.rmtree(AR_DIR, ignore_errors=True)
    os.makedirs(AR_DIR)
    total = 0
    for y in YEARS:
        # C2C：CRSP CIZ 日频。
        c = pd.read_parquet(f"{DATA}/crsp_daily_{y}.parquet",
                            columns=["permno", "dlycaldt", "dlyret"]).dropna(subset=["dlyret"])
        c = c.rename(columns={"dlycaldt": "date", "dlyret": "ret_c2c"})
        c["date"] = pd.to_datetime(c["date"])
        c["permno"] = c["permno"].astype("int64")

        # O2O：yifei v5，整条 O2O residual / factor 线用的同一份收益
        o = pq.read_table(RET_PANEL, columns=["PERMNO", "date", "O2O_RET"],
                          filters=[("date", ">=", datetime.date(y, 1, 1)),
                                   ("date", "<=", datetime.date(y, 12, 31))]).to_pandas()
        o = o.dropna(subset=["O2O_RET"]).rename(columns={"PERMNO": "permno", "O2O_RET": "ret_o2o"})
        o["date"] = pd.to_datetime(o["date"])
        o["permno"] = o["permno"].astype("int64")

        d = c.merge(o, on=["permno", "date"], how="outer")
        d = d[d["date"].dt.year == y]
        d["td_idx"] = d["date"].map(cal_map)
        d = d.dropna(subset=["td_idx"])                       # 非交易日（理论上没有）
        d["ffyear"] = ffyear_of(d["date"]).astype("int32")
        d = d.merge(mem, on=["permno", "ffyear"], how="left")
        d["port25"] = d["port25"].fillna(0).astype("int8")     # 0 = 当年没归上组
        d = d.sort_values(["date", "permno"])

        ti = d["td_idx"].to_numpy().astype("int64")
        p = d["port25"].to_numpy().astype(int)
        bench_r, ar = {}, {}
        for tag in ("c2c", "o2o"):
            # port25=0 的行取到 BRET 的 0 行（全 NaN），bench 与 ar 自然都是 NaN
            bench_r[tag] = BRET[tag][p, ti]
            ar[tag] = d[f"ret_{tag}"].to_numpy(dtype="float64") - bench_r[tag]

        pq.write_table(pa.Table.from_arrays(
            [pa.array(d["permno"].to_numpy(), pa.int32()),
             pa.array(d["date"].to_numpy().astype("datetime64[s]")),
             pa.array(ti, pa.int32()),
             pa.array(d["port25"].to_numpy(), pa.int8()),
             pa.array(d["ret_c2c"].to_numpy(dtype="float64"), pa.float32()),
             pa.array(d["ret_o2o"].to_numpy(dtype="float64"), pa.float32()),
             pa.array(bench_r["c2c"], pa.float32()),
             pa.array(bench_r["o2o"], pa.float32()),
             pa.array(ar["c2c"], pa.float32()),
             pa.array(ar["o2o"], pa.float32())], schema=SCHEMA),
            f"{AR_DIR}/ar_daily_{y}.parquet", compression="zstd")
        total += len(d)
        print(f"  {y}: {len(d):>9,} 行 (C2C {d['ret_c2c'].notna().mean():.3f}, "
              f"O2O {d['ret_o2o'].notna().mean():.3f}, port25 {(d['port25'] > 0).mean():.3f})",
              flush=True)

    size = sum(os.path.getsize(f) for f in glob.glob(f"{AR_DIR}/ar_daily_*.parquet"))
    print(f"\n→ {AR_DIR}/  {total:,} 行，{len(YEARS)} 个年度文件，{size / 1e9:.2f} GB")


  1996: 2,220,562 行 (C2C 1.000, O2O 0.996, port25 0.672)
  1997: 2,296,700 行 (C2C 1.000, O2O 0.999, port25 0.674)
  1998: 2,260,650 行 (C2C 1.000, O2O 0.999, port25 0.681)
  1999: 2,139,080 行 (C2C 1.000, O2O 0.999, port25 0.680)
  2000: 2,093,858 行 (C2C 1.000, O2O 0.999, port25 0.666)
  2001: 1,920,710 行 (C2C 1.000, O2O 0.999, port25 0.683)
  2002: 1,816,107 行 (C2C 1.000, O2O 0.999, port25 0.688)
  2003: 1,712,676 行 (C2C 1.000, O2O 0.999, port25 0.677)
  2004: 1,687,978 行 (C2C 1.000, O2O 0.999, port25 0.652)
  2005: 1,697,106 行 (C2C 1.000, O2O 0.999, port25 0.633)
  2006: 1,700,044 行 (C2C 1.000, O2O 0.998, port25 0.620)
  2007: 1,741,243 行 (C2C 1.000, O2O 0.990, port25 0.592)
  2008: 1,752,185 行 (C2C 1.000, O2O 0.950, port25 0.582)
  2009: 1,666,595 行 (C2C 1.000, O2O 0.888, port25 0.586)
  2010: 1,658,455 行 (C2C 1.000, O2O 0.870, port25 0.556)
  2011: 1,681,326 行 (C2C 1.000, O2O 0.847, port25 0.526)
  2012: 1,676,891 行 (C2C 1.000, O2O 0.829, port25 0.509)
  2013: 1,685,763 行 (C2C 1.000,

In [10]:
# check：逐年行数与覆盖率
rec = []
for f in sorted(glob.glob(f"{AR_DIR}/ar_daily_*.parquet")):
    t = pq.read_table(f, columns=["port25", "ret_c2c", "ret_o2o", "ar_c2c", "ar_o2o"]).to_pandas()
    rec.append({"year": int(os.path.basename(f)[9:13]), "n": len(t),
                "有port25": (t["port25"] > 0).mean(),
                "有ret_c2c": t["ret_c2c"].notna().mean(),
                "有ret_o2o": t["ret_o2o"].notna().mean(),
                "有ar_c2c": t["ar_c2c"].notna().mean(),
                "有ar_o2o": t["ar_o2o"].notna().mean()})
    del t
chk = pd.DataFrame(rec).set_index("year")
print(chk.round(3).to_string())
print(f"\n合计 {chk['n'].sum():,} 行")


            n  有port25  有ret_c2c  有ret_o2o  有ar_c2c  有ar_o2o
year                                                        
1996  2220562    0.672       1.0     0.996    0.672    0.669
1997  2296700    0.674       1.0     0.999    0.674    0.673
1998  2260650    0.681       1.0     0.999    0.681    0.680
1999  2139080    0.680       1.0     0.999    0.680    0.679
2000  2093858    0.666       1.0     0.999    0.666    0.665
2001  1920710    0.683       1.0     0.999    0.683    0.683
2002  1816107    0.688       1.0     0.999    0.688    0.688
2003  1712676    0.677       1.0     0.999    0.677    0.676
2004  1687978    0.652       1.0     0.999    0.652    0.652
2005  1697106    0.633       1.0     0.999    0.633    0.633
2006  1700044    0.620       1.0     0.998    0.620    0.620
2007  1741243    0.592       1.0     0.990    0.592    0.590
2008  1752185    0.582       1.0     0.950    0.582    0.580
2009  1666595    0.586       1.0     0.888    0.586    0.585
2010  1658455    0.556  

In [11]:
# 与 car.parquet 对照：用本表的日收益重算 2010 年公告事件的 CAR[0,1]，应当逐个对上。
# 顺带演示 buy-and-hold 的算法 —— 连乘个股、连乘基准、再相减，缺失日按收益 0 处理。
YR = 2010
ret = pd.concat([pd.read_parquet(f"{AR_DIR}/ar_daily_{y}.parquet",
                                 columns=["permno", "td_idx", "ret_c2c", "ret_o2o"])
                 for y in (YR, YR + 1)], ignore_index=True).set_index(["permno", "td_idx"])

ev = pd.read_parquet(f"{BUILD}/car.parquet",
                     columns=["permno", "td0", "td0_idx", "port25",
                              "stk_ann_c2c", "car_ann_c2c", "stk_ann_o2o", "car_ann_o2o"])
ev = ev[(ev["td0"].dt.year == YR) & ev["td0_idx"].notna()].copy()
ev["td0_idx"] = ev["td0_idx"].astype("int64")
p = ev["port25"].fillna(0).to_numpy().astype(int)
t0 = ev["td0_idx"].to_numpy()


def leg(col, k):
    idx = pd.MultiIndex.from_arrays([ev["permno"].astype("int64"), ev["td0_idx"] + k])
    return np.nan_to_num(ret[col].reindex(idx).to_numpy())


for tag in ("c2c", "o2o"):
    stk = (1 + leg(f"ret_{tag}", 0)) * (1 + leg(f"ret_{tag}", 1)) - 1
    b0 = np.nan_to_num(BRET[tag][p, t0])
    b1 = np.nan_to_num(BRET[tag][p, t0 + 1])
    car = np.where(p > 0, stk - ((1 + b0) * (1 + b1) - 1), np.nan)
    ok = ev[f"car_ann_{tag}"].notna().to_numpy()
    ref = ev[f"car_ann_{tag}"].to_numpy()[ok]
    print(f"{tag}: 对照 {ok.sum():,} 个事件 | "
          f"stk 最大差 {np.nanmax(np.abs(stk[ok] - ev[f'stk_ann_{tag}'].to_numpy()[ok])):.2e} | "
          f"car 最大差 {np.nanmax(np.abs(car[ok] - ref)):.2e} | "
          f"相关 {np.corrcoef(car[ok], ref)[0, 1]:.6f}")


c2c: 对照 13,941 个事件 | stk 最大差 1.99e-07 | car 最大差 1.99e-07 | 相关 1.000000
o2o: 对照 13,932 个事件 | stk 最大差 1.75e-07 | car 最大差 1.75e-07 | 相关 1.000000


## build/pead_panel.parquet

In [12]:
import pandas as pd
df = pd.read_parquet("build/pead_panel.parquet")
df.head()

,eid,permno,ticker,anndats,td0,td0_idx,pends,actual_eps,anntims,lag,...,io_stale_days,datadate,evol,epersist,eps_stale_days,lag2,lag3,month,dow,qtr
0,0,75554,SEM1,1995-04-01,1996-01-02,0,1995-12-31,-0.15,11:25:00,-274,...,NaN,1995-03-31,0.215703,0.050927,1.0,75076,-20570824,4,6,1995Q2
1,1,16791,BKNT,1995-06-09,1996-01-02,0,1995-09-30,<NA>,00:00:00,-113,...,NaN,1995-04-30,0.031552,-0.461407,40.0,12769,-1442897,6,5,1995Q2
2,2,79713,MAXM,1995-08-02,1996-01-02,0,1995-07-31,0.14,00:00:00,2,...,NaN,1995-06-30,0.078102,-0.148101,33.0,4,8,8,3,1995Q3
3,3,64697,PLFC,1995-08-04,1996-01-02,0,1995-07-31,-0.21,00:00:00,4,...,NaN,1995-07-31,0.275292,-0.219130,4.0,16,64,8,5,1995Q3
4,4,13100,MA,1995-08-07,1996-01-02,0,1995-07-31,0.3533,00:00:00,7,...,38.0,1995-07-31,0.364214,-0.619047,7.0,49,343,8,1,1995Q3


In [13]:
ev = df[~df["flag_pre_calendar"]]
ev.head()

,eid,permno,ticker,anndats,td0,td0_idx,pends,actual_eps,anntims,lag,...,io_stale_days,datadate,evol,epersist,eps_stale_days,lag2,lag3,month,dow,qtr
5628,5628,19502,WAG,1996-01-02,1996-01-02,0,1995-11-30,0.065,00:00:00,33,...,2.0,1995-11-30,0.003068,-0.056539,33.0,1089,35937,1,2,1996Q1
5629,5629,65008,ASYS,1996-01-02,1996-01-02,0,1995-09-30,0.11,00:00:00,94,...,NaN,1995-12-31,0.378390,0.331170,2.0,8836,830584,1,2,1996Q1
5630,5630,66262,RLIF,1996-01-02,1996-01-02,0,1995-09-30,<NA>,00:00:00,94,...,NaN,NaT,NaN,NaN,NaN,8836,830584,1,2,1996Q1
5631,5631,68638,PP,1996-01-02,1996-01-02,0,1995-09-30,-0.16,00:00:00,94,...,NaN,1995-12-31,0.387623,-0.139369,2.0,8836,830584,1,2,1996Q1
5632,5632,76687,RAG1,1996-01-02,1996-01-02,0,1995-11-30,0.1524,00:00:00,33,...,NaN,1995-11-30,0.093971,-0.263283,33.0,1089,35937,1,2,1996Q1


# 合并1996-2026的AR大表

In [14]:
# check AR 某年表sample
import pandas as pd
df_ar = pd.read_parquet("build/ar_daily/ar_daily_1996.parquet")
df_ar.tail()

,permno,date,td_idx,port25,ret_c2c,ret_o2o,bench_c2c,bench_o2o,ar_c2c,ar_o2o
2220557,93009,1996-12-31,253,1,0.000000,-7.142857e-02,0.020380,-0.010269,-0.020380,-0.061159
2220558,93025,1996-12-31,253,2,-0.041667,1.499996e-01,0.018618,-0.004195,-0.060285,0.154195
2220559,93105,1996-12-31,253,4,0.021739,2.173900e-02,0.009269,0.000315,0.012470,0.021424
2220560,93236,1996-12-31,253,1,-0.047619,5.000000e-08,0.020380,-0.010269,-0.067999,0.010269
2220561,93316,1996-12-31,253,3,-0.015385,-3.906250e-07,0.008542,-0.006304,-0.023927,0.006304


In [5]:
# 合并成单文件
import glob
import os

import pyarrow.parquet as pq

ROOT = "/project/dachxiu/zan/PEAD"
AR_DIR = f"{ROOT}/build/ar_daily"
OUT = f"{AR_DIR}/ar_daily.parquet"         # 合并位置：年度文件同一个目录

files = sorted(glob.glob(f"{AR_DIR}/ar_daily_*.parquet"))
print(f"待合并 {len(files)} 个文件: {os.path.basename(files[0])} ~ {os.path.basename(files[-1])}")

w, total = None, 0
try:
    for f in files:
        t = pq.read_table(f)
        if w is None:
            w = pq.ParquetWriter(OUT, t.schema, compression="zstd")
        w.write_table(t)
        total += t.num_rows
        del t
finally:
    if w is not None:
        w.close()

# 时间范围从文件自身的 row group statistics 读，不读数据
md = pq.ParquetFile(OUT).metadata
names = md.schema.names
lo = {c: None for c in ("date", "td_idx")}
hi = {c: None for c in ("date", "td_idx")}
for g in range(md.num_row_groups):
    for c in lo:
        s = md.row_group(g).column(names.index(c)).statistics
        if s is not None:
            lo[c] = s.min if lo[c] is None else min(lo[c], s.min)
            hi[c] = s.max if hi[c] is None else max(hi[c], s.max)

npermno = len(pq.read_table(OUT, columns=["permno"]).column(0).unique())
print(f"\n→ {OUT}")
print(f"   {md.num_rows:,} 行，{os.path.getsize(OUT) / 1e9:.3f} GB，{md.num_row_groups} 个 row group")
print(f"   date   {lo['date'].date()} ~ {hi['date'].date()}")
print(f"   td_idx {lo['td_idx']} ~ {hi['td_idx']}（共 {hi['td_idx'] - lo['td_idx'] + 1:,} 个交易日）")
print(f"   permno {npermno:,} 个")
print(f"   写入行数 {total:,}，与文件一致: {total == md.num_rows}")
print(pq.ParquetFile(OUT).schema_arrow)


待合并 31 个文件: ar_daily_1996.parquet ~ ar_daily_2026.parquet

→ /project/dachxiu/zan/PEAD/build/ar_daily/ar_daily.parquet
   59,487,271 行，0.918 GB，71 个 row group
   date   1996-01-02 ~ 2026-06-30
   td_idx 0 ~ 7672（共 7,673 个交易日）
   permno 29,876 个
   写入行数 59,487,271，与文件一致: True
permno: int32
date: timestamp[ms]
td_idx: int32
port25: int8
ret_c2c: float
ret_o2o: float
bench_c2c: float
bench_o2o: float
ar_c2c: float
ar_o2o: float
